In [1]:
from pandas._typing import F
import pandas as pd

# Đường dẫn tới file gốc của bạn
file_path = r"C:\Users\User\Desktop\R221_Nhật ký bán hàng.xlsx"

# Đọc tất cả các sheet vào một từ điển (dictionary)
# Dùng header=None để tránh việc Pandas lấy nhầm dòng rác làm tiêu đề
all_sheets = pd.read_excel(file_path, sheet_name=None, header=None)

df_list = []

for sheet_name, df in all_sheets.items():
    # Tìm dòng chứa chữ "STT" (ở cột đầu tiên) để xác định tiêu đề chuẩn
    header_row = df[df[0] == 'STT'].index.min()
    
    if pd.isna(header_row):
        continue  # Bỏ qua sheet nếu không có cấu trúc đúng
        
    # Lấy dữ liệu từ dòng tiêu đề trở xuống
    df_cleaned = df.iloc[header_row + 1:].copy()
    df_cleaned.columns = df.iloc[header_row]  # Gán lại tên cột
    
    # Các dòng Total thường chứa mã khách hàng/text thay vì số thứ tự.
    # Ép kiểu cột STT về dạng số, các dòng text sẽ bị biến thành NaN (Not a Number)
    df_cleaned['STT_numeric'] = pd.to_numeric(df_cleaned['STT'], errors='coerce')
    
    # Loại bỏ các dòng bị NaN (chính là các dòng Total và dòng rác)
    df_cleaned = df_cleaned.dropna(subset=['STT_numeric']).copy()
    
    # Xóa cột STT_numeric tạm thời
    df_cleaned.drop(columns=['STT_numeric'], inplace=True)
    
    df_list.append(df_cleaned)

# Gom tất cả các sheet thành 1 bảng duy nhất
final_df = pd.concat(df_list, ignore_index=True)

# Đánh lại số thứ tự cho liền mạch từ đầu đến cuối
final_df['STT'] = range(1, len(final_df) + 1)

# Xuất kết quả ra file Excel tổng hợp
output_path = r"C:\Users\User\Desktop\R221_Nhat_Ky_Ban_Hang_Gop.xlsx"
final_df.to_excel(output_path, index=False)
print(f"Đã gộp thành công! Dữ liệu được lưu tại: {output_path}")

Đã gộp thành công! Dữ liệu được lưu tại: C:\Users\User\Desktop\R221_Nhat_Ky_Ban_Hang_Gop.xlsx
